# Anime

In [46]:
import pandas as pd
import numpy as np
import plotly.express as px

In [47]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import plotly.graph_objects as go
import ast
import plotly.express as px
import pycountry

In [48]:
# load dataset
df_profiles = pd.read_csv("../../data/cleaned/profiles_cleaned.csv")
df_person_anime_works = pd.read_csv("../../data/cleaned/person_anime_works_cleaned.csv")
df_person_details = pd.read_csv("../../data/cleaned/person_details_cleaned.csv")
df_person_alternate_names = pd.read_csv("../../data/cleaned/person_alternate_names_cleaned.csv")



In [49]:
df_person_details["birthday"] = pd.to_datetime(df_person_details["birthday"], errors="coerce")
df_person_details["birth_year"] = df_person_details["birthday"].dt.year.astype("Int64")

df_person_details["favorites"] = pd.to_numeric(df_person_details["favorites"], errors="coerce")


In [50]:
#Profiles
print("Righe totali profiles:", len(df_profiles))
print("Utenti unici (username):", df_profiles["username"].nunique())
print("Location non-missing:", df_profiles["location"].notna().mean().round(3), "→ quota")
print("Joined non-missing:", df_profiles["joined"].notna().mean().round(3), "→ quota")

profiles = df_profiles

Righe totali profiles: 337154
Utenti unici (username): 337154
Location non-missing: 1.0 → quota
Joined non-missing: 0.995 → quota


## In quale aree del mondo gli anime sono maggiormente seguiti?


In [51]:
profiles = df_profiles.copy()

# pulizia base location
profiles["location"] = profiles["location"].astype("string").str.strip()
profiles.loc[
    profiles["location"].str.lower().isin(["", "nan", "none", "unknown"]),
    "location"
] = pd.NA
#estraggo paese
profiles["country_raw"] = profiles["location"].str.split(",").str[-1].str.strip()

#
country_fix = {
    "Usa": "United States",
    "US": "United States",
    "United States Of America": "United States",
    "UK": "United Kingdom",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Russian Federation": "Russia",
    "Korea, Republic of": "South Korea",
    "Korea, Rep.": "South Korea",
    "Turkey": "Türkiye",
    "Turkiye": "Türkiye",
    "Türkiye": "Türkiye",

}
profiles["country_clean"] = profiles["country_raw"].replace(country_fix)
profiles["country_clean"] = profiles["country_clean"].where(
    profiles["country_clean"].isna(),
    profiles["country_clean"].str.replace(r"\s+", " ", regex=True).str.strip()
)

# ISO3 mapping
def to_iso3(x):
    if pd.isna(x):
        return pd.NA
    try:
        return pycountry.countries.lookup(x).alpha_3
    except Exception:
        return pd.NA

profiles["iso3"] = profiles["country_clean"].apply(to_iso3)

#controlli utili per verificare che ho preso tutti i paesi
u_total = profiles["username"].nunique()
unmapped = profiles.loc[profiles["iso3"].isna(), "username"].nunique()
coverage_iso3 = 1 - (unmapped / u_total)

print(f"Profiles coverage: {len(profiles):,} rows | {u_total:,} unique users")
print(f"Map coverage (ISO3): {coverage_iso3:.3f} | unmapped users: {unmapped}")


# conteggi
country_counts = (
    profiles.dropna(subset=["iso3"])
            .groupby("iso3")["username"]
            .nunique()
            .reset_index(name="n_users")
)
country_counts["log_users"] = np.log10(country_counts["n_users"] + 1)

# mappa
fig = px.choropleth(
    country_counts,
    locations="iso3",
    locationmode="ISO-3",
    color="log_users",
    hover_data={"n_users": True, "log_users": ":.2f"},
    title="Dove gli anime sono maggiormente seguiti?"
)
fig.update_layout(
    template="plotly_white",
    width=1100,
    height=600,
    coloraxis_colorbar=dict(title="log10(utenti)", tickformat=".1f")
)
fig.update_geos(showland=True, landcolor="lightgrey", showocean=True, oceancolor="#e6f2ff")
fig.show()

Profiles coverage: 337,154 rows | 337,154 unique users
Map coverage (ISO3): 1.000 | unmapped users: 0


La mappa evidenzia paesi in cui il dataset registra una maggiore presenza di utenti appassionati di anime. L'intensità del colore cresce all'aumentare del numero di utenti unici per paese; l'uso della trasformazione log(n_users + 1) comprime le differenze estreme di valore e rende visibili anche paesi con valori più bassi.

#### A questo punto possiamo vedere come l'interesse per gli anime si è sviluppato durante il corso del tempo, considerando i paesi dove esse son maggiormente diffuse.

## Come si è evoluta nel tempo la crescita della community/nuovi utenti per gli anime nelle aree dove sono più concentrate?

In [52]:
df = df_profiles.copy()

# converto joined + extract year
profiles["joined"] = pd.to_datetime(profiles["joined"], errors="coerce")
profiles["joined_year"] = profiles["joined"].dt.year

#uso la stessa logica del grafico precedente
profiles["location"] = profiles["location"].astype("string").str.strip()
profiles.loc[
    profiles["location"].str.lower().isin(["", "nan", "none", "unknown"]),
    "location"
] = pd.NA

profiles["country_raw"] = profiles["location"].str.split(",").str[-1].str.strip()

country_fix = {
    "Usa": "United States",
    "US": "United States",
    "United States Of America": "United States",
    "UK": "United Kingdom",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Russian Federation": "Russia",
    "Korea, Republic of": "South Korea",
    "Korea, Rep.": "South Korea",
    "Turkey": "Türkiye",
    "Turkiye": "Türkiye",
    "Türkiye": "Türkiye",
}
profiles["country_clean"] = profiles["country_raw"].replace(country_fix)

profiles["country_clean"] = profiles["country_clean"].where(
    profiles["country_clean"].isna(),
    profiles["country_clean"].str.replace(r"\s+", " ", regex=True).str.strip()
)

# filtri sugli anni non sensati
profiles = profiles.dropna(subset=["joined_year", "country_clean"])
profiles = profiles[profiles["joined_year"].between(1995, 2024)]

#top 6 countries
top_countries = (
    profiles.groupby("country_clean")["username"]
            .nunique()
            .sort_values(ascending=False)
            .head(6)
            .index
)

profiles_top = profiles[profiles["country_clean"].isin(top_countries)].copy()

# new users per year
trend = (
    profiles_top.groupby(["joined_year", "country_clean"])["username"]
                .nunique()
                .reset_index(name="n_users")
)


#line chart
fig = px.line(
    trend,
    x="joined_year",
    y="n_users",
    color="country_clean",
    markers=True,
    title="Crescita della community nel tempo: nuovi utenti registrati per anno (top paesi)"
)

fig.update_layout(
    template="plotly_white",
    width=1100, height=600,
    xaxis_title="Anno",
    yaxis_title="Nuovi utenti" ,
    legend_title_text = "Paese"
)

fig.show()


Il grafico mostra l'andamento dei nuovi utenti registrati ogni anno nei 6 paesi con la maggiore concentrazione di utenti nel dataset. Il Giappone e gli Stati Uniti offrono contributo dominante, mentre gli altri paesi hanno volumi più contenuti ma comunque trend simili.

#### E l'evoluzione a livello globale? Vediamo come si modifica a livello temporale ...

## Crescita della community nel tempo a livello globale

In [53]:
df = df_profiles.copy()
#convert joined to datetime and extract  year
df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
df["joined_year"] = df["joined"].dt.year

users_per_year = (
    df.dropna(subset=["joined_year"])
      .assign(joined_year=lambda x: x["joined_year"].astype(int))
      .groupby("joined_year", observed=True)["username"]
      .nunique()
      .reset_index(name="new_users")
      .sort_values("joined_year")
)

users_per_year = users_per_year[users_per_year["joined_year"].between(2005, 2024)].copy()

#line chart
fig = px.line(
    users_per_year,
    x="joined_year",
    y="new_users",
    markers=True,
    title="Crescita della community nel tempo: nuovi utenti registrati per anno",
    labels={"joined_year": "Anno", "new_users": "Nuovi utenti"},
)
fig.update_layout(template="plotly_white", width=1100, height=600)
fig.show()

Il grafico mostra l’andamento dei nuovi utenti registrati ogni anno: si osserva una crescita nel tempo, con picchi marcati negli anni più recenti.

Dopo aver osservato la crescita della community nel tempo, analizziamo come sono composte le liste degli utenti, distinguendo  l'anno di registrazione, per capire se le abitudini di consumo variano tra utenti ‘vecchi’ e ‘nuovi’.

## Composizione delle liste per anno di iscrizione:

In [54]:
df = df_profiles.copy()

# anno iscrizione
df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
df["joined_year"] = df["joined"].dt.year

# colonne di consumo
cols = ["watching", "completed", "on_hold", "dropped", "plan_to_watch"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0)


# aggregazione per anno
consumption_time = (
    df.dropna(subset=["joined_year"])
      .assign(joined_year=lambda x: x["joined_year"].astype(int))
      .groupby("joined_year", observed=True)[cols]
      .sum()
      .reset_index()
      .sort_values("joined_year")
)
#periodo
consumption_time = consumption_time[consumption_time["joined_year"].between(2005, 2024)].copy()

#trasformo in percentuale, così da avere dati più omogenei
den = consumption_time[cols].sum(axis=1)
consumption_time = consumption_time[den > 0].copy()
den = consumption_time[cols].sum(axis=1)

consumption_pct = consumption_time.copy()
consumption_pct[cols] = consumption_pct[cols].div(den, axis=0) * 100

# righe controllo
print(f"joined_year coverage: {df['joined_year'].notna().mean():.3f}")
print(f"cohort share sums (min–max): {consumption_pct[cols].sum(axis=1).min():.1f}% – {consumption_pct[cols].sum(axis=1).max():.1f}%")


#grafico
consumption_long = consumption_pct.melt(
    id_vars="joined_year",
    value_vars=cols,
    var_name="status",
    value_name="share"
)
#ordine
status_order = ["completed", "plan_to_watch", "watching", "on_hold", "dropped"]

fig = px.area(
    consumption_long,
    x="joined_year",
    y="share",
    color="status",
    title="Composizione delle liste per coorte di iscrizione",
    labels={"joined_year": "Anno di iscrizione", "share": "Quota (%)", "status": "Status"},
    category_orders={"status": status_order}
)

fig.update_layout(template="plotly_white", width=1100, height=600)
fig.update_yaxes(ticksuffix="%", range=[0,100], dtick=10)
fig.update_traces(hovertemplate="Quota=%{y:.1f}%<br>Anno=%{x}<extra></extra>")
fig.show()

joined_year coverage: 0.995
cohort share sums (min–max): 100.0% – 100.0%


Il grafico mostra, per ogni coorte di iscrizione (anno di registrazione), la composizione percentuale delle liste degli utenti nei diversi status (watching, completed, on_hold, dropped, plan_to_watch). Le quote sommano a 100% per anno e permettono di confrontare come cambia la struttura delle liste tra coorti.

### Quanti anime sono state completate ogni anno?

In [55]:
#person_details
details = df_person_details.copy()

print("Righe totali:", len(details))
print("Person ID missing:", details["person_mal_id"].isna().mean().round(3), "→ quota")
print("Person ID unici:", details["person_mal_id"].nunique() if "person_mal_id" in details.columns else "N/A")


Righe totali: 76594
Person ID missing: 0.0 → quota
Person ID unici: 76594


In [56]:
#utilizzo da qui in poi dataset person_anime_works
works = df_person_anime_works.copy()

rows = len(works)
p = works["person_mal_id"].nunique()
a = works["anime_mal_id"].nunique()
pos_missing = works["position"].isna().mean()

print(f"Anime works coverage: {rows:,} rows | {p:,} persons | {a:,} anime")
print(f"Position completeness: {(1-pos_missing):.3f} | unique roles (raw): {works['position'].nunique(dropna=True):,}")

Anime works coverage: 458,091 rows | 26,236 persons | 20,148 anime
Position completeness: 1.000 | unique roles (raw): 80,036


CI spostiamo ora sull'analisi degli utenti che lavorano nell'industria anime, analizzando quali sono i lavori più diffusi (top 10), confrontandoli con la loro notorietà

## Vediamo ora se i ruoli più frequenti sono anche i preferiti dell'industria anime:

In [58]:
works = df_person_anime_works.copy()
details = df_person_details[["person_mal_id", "favorites"]].copy()

# ruolo
works["position_norm"] = (
    works["position"].astype("string")
    .str.replace(r"\s*\(ep\s*\d+\)\s*$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
works.loc[works["position_norm"].str.lower().isin(["", "nan", "none", "unknown"]), "position_norm"] = pd.NA

# conteggi occorrenze
role_counts = works["position_norm"].dropna().value_counts().reset_index()
role_counts.columns = ["role", "occurrences"]

top10 = role_counts.head(10).copy()
total_occ = role_counts["occurrences"].sum()
top10["occ_pct"] = top10["occurrences"] / total_occ * 100

# favorites
details["person_mal_id"] = pd.to_numeric(details["person_mal_id"], errors="coerce").astype("Int64")
details["favorites"] = pd.to_numeric(details["favorites"], errors="coerce").fillna(0)
details = details.drop_duplicates(subset=["person_mal_id"])

# persone uniche per ruolo + merge favorites
role_people = (
    works.loc[works["position_norm"].isin(top10["role"]), ["person_mal_id", "position_norm"]]
    .dropna(subset=["person_mal_id", "position_norm"])
    .drop_duplicates()
    .rename(columns={"position_norm": "role"})
).merge(details, on="person_mal_id", how="left")

role_people["favorites"] = role_people["favorites"].fillna(0)

fav_stats = (
    role_people.groupby("role", observed=True)["favorites"]
    .agg(
        people="count",
        share_nonzero=lambda s: (s > 0).mean() * 100,
        median_fav_pos=lambda s: s[s > 0].median(),
    )
    .reset_index()
)
fav_stats["median_fav_pos"] = fav_stats["median_fav_pos"].fillna(0)

top10 = top10.merge(fav_stats, on="role", how="left")
role_order = top10.sort_values("occurrences", ascending=False)["role"].tolist()
top10 = top10.set_index("role").loc[role_order].reset_index()  # riallinea tutto in modo sicuro

role_color_map = {
    "Key Animation": "#7fd6c2",
    "Animation Director": "#fff7a8",
    "Producer": "#c7c2e0",
    "In-Between Animation": "#ff7f7a",
    "2nd Key Animation": "#7fb3d5",
    "Episode Director": "#f5b041",
    "Planning": "#a8df65",
    "Original Creator": "#f7b6d2",
    "Music": "#d9d9d9",
    "Director": "#af7ac5",
}
occ_colors = [role_color_map.get(r, "#7fd6c2") for r in top10["role"]]


y = top10["role"].tolist()

text_occ = top10["occ_pct"].round(1).astype(str) + "%"
text_fav = top10["median_fav_pos"].round(0).astype(int).astype(str)

custom = np.stack(
    [top10["occurrences"], top10["people"], top10["share_nonzero"], top10["median_fav_pos"]],
    axis=1
)

hover = (
    "Ruolo=%{y}<br>"
    "Occorrenze=%{customdata[0]}<br>"
    "Persone (nel ruolo)=%{customdata[1]}<br>"
    "% con favorites>0=%{customdata[2]:.1f}%<br>"
    "Mediana favorites (>0)=%{customdata[3]:.0f}"
    "<extra></extra>"
)

fig = go.Figure()

# Occorrenze
fig.add_trace(go.Bar(
    x=top10["occurrences"], y=y, orientation="h",
    text=text_occ, textposition="outside",
    marker=dict(color=occ_colors),
    customdata=custom, hovertemplate=hover,
    visible=True
))

# Favorites (colore unico)
fig.add_trace(go.Bar(
    x=top10["median_fav_pos"],
    y=y,
    orientation="h",
    text=text_fav,
    textposition="outside",
    marker=dict(color=occ_colors),
    customdata=custom,
    hovertemplate=hover,
    visible=False
))

fig.update_layout(
    template="plotly_dark",
    width=1100, height=650,
    showlegend=False,
    title="Quali sono i ruoli più diffusi nell'industria degli anime (Top 10)?",
    xaxis_title="Numero di occorrenze",
    yaxis_title="Ruolo",
    yaxis=dict(categoryorder="array", categoryarray=role_order),
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.01, y=1.07,
        showactive=True,
        bgcolor="rgba(255,255,255,0.92)",
        bordercolor="rgba(0,0,0,0.25)",
        font=dict(color="black"),
        buttons=[
            dict(
                label="Ruoli più diffusi (occorrenze)",
                method="update",
                args=[{"visible": [True, False]},
                      {"title.text": "Quali sono i ruoli più diffusi nell'industria degli anime (Top 10)?",
                "xaxis.title.text": "Numero di occorrenze",}],
            ),
            dict(
                label="Notorietà (mediana favorites > 0)",
                method="update",
                args=[{"visible": [False, True]},
                      {"title.text": "Notorietà per ruolo: sui Top 10 ruoli",
                "xaxis.title.text": "Mediana favorites" }], # solo >0
            ),
        ],
    )],
)

fig.show()

## Quali sono le generazioni più diffuse nell'industria anime?

In [59]:
# prendo person_mal_id
works_people = (
    df_person_anime_works[["person_mal_id"]]
    .copy()
)
#converto in numerico e tolgo righe mancanti o duplicati (controllo ch eho aggiunto per sicurezza)
works_people = df_person_anime_works[["person_mal_id"]].copy()
works_people["person_mal_id"] = pd.to_numeric(works_people["person_mal_id"], errors="coerce").astype("Int64")
works_people = works_people.dropna(subset=["person_mal_id"]).drop_duplicates()

# recupero anni di nascita da details
details_birth = df_person_details[["person_mal_id", "birthday"]].copy()
details_birth["person_mal_id"] = pd.to_numeric(details_birth["person_mal_id"], errors="coerce").astype("Int64")
details_birth["birthday"] = pd.to_datetime(details_birth["birthday"], errors="coerce", utc=True)
details_birth["birth_year"] = details_birth["birthday"].dt.year.astype("Int64")

# filtro anni (prima di 1900 e dopo 2024 gli anni li considero 'non palusibili' per evitare dati strani)
details_birth.loc[~details_birth["birth_year"].between(1900, 2024), "birth_year"] = pd.NA
details_birth = details_birth.drop_duplicates(subset=["person_mal_id"])

staff = works_people.merge(details_birth[["person_mal_id", "birth_year"]], on="person_mal_id", how="left")

#controlli
tot_people = staff["person_mal_id"].nunique()
known_people = staff.loc[staff["birth_year"].notna(), "person_mal_id"].nunique()
print(f"Staff in credits: {tot_people:,} unique people")
print(f"Birth year coverage: {known_people/tot_people:.3f} ({known_people:,}/{tot_people:,})")


# generazioni
labels = ["Silent Generation", "Boomers", "Gen X", "Millennials", "Gen Z", "Gen Alpha"]
staff["generation"] = pd.cut(
    staff["birth_year"],
    bins=[-np.inf, 1945, 1964, 1980, 1996, 2012, 2024],
    labels=labels,
    right=True
)
gen_counts = (
    staff.dropna(subset=["generation"])
         .groupby("generation", observed=True)["person_mal_id"]
         .nunique()
         .reset_index(name="count")
)

gen_counts["generation"] = pd.Categorical(gen_counts["generation"], categories=labels, ordered=True)
gen_counts = gen_counts.sort_values("generation")
gen_counts["share"] = gen_counts["count"] / gen_counts["count"].sum() * 100

color_map = {
    "Silent Generation": "#A0AEC0",
    "Boomers": "#F6AD55",
    "Gen X": "#63B3ED",
    "Millennials": "#68D391",
    "Gen Z": "#FC8181",
    "Gen Alpha": "#B794F4",
}

fig = px.bar(
    gen_counts,
    x="generation",
    y="count",
    color="generation",
    color_discrete_map=color_map,
    text=gen_counts["share"].round(1).astype(str) + "%",
    title="Quali generazioni sono più diffuse tra le persone che fanno parte dell’industria anime?",
    labels={"count": "Numero di persone", "generation": "Generazione"},
)
fig.update_layout(template="plotly_white", width=1100, height=600, showlegend=False)
fig.update_traces(textposition="outside")
fig.update_yaxes(rangemode="tozero")
fig.show()

Staff in credits: 26,236 unique people
Birth year coverage: 0.244 (6,409/26,236)


Il grafico mostra la distribuzione delle generazioni tra le persone presenti nello staff, ogni barra evidenzia il numero di persone uniche per generazione, con la percentuale calcolata sul sottoinsieme per cui l'anno di nascita è disponibile.

## In quali anni di nascita si concentrano le diverse generazioni dell'industria anime?

In [60]:
def print_columns(name, df):
    cols = list(df.columns)
    print(f"\n{name} — shape={df.shape} — {len(cols)} colonne")
    print(cols)

print_columns("profiles", df_profiles)
print_columns("person_anime_works", df_person_anime_works)
print_columns("person_details", df_person_details)
print_columns("person_alternate_names", df_person_alternate_names)  # o df_person_alternate_names


profiles — shape=(337154, 10) — 10 colonne
['username', 'gender', 'birthday', 'location', 'joined', 'watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']

person_anime_works — shape=(458091, 3) — 3 colonne
['person_mal_id', 'position', 'anime_mal_id']

person_details — shape=(76594, 11) — 11 colonne
['person_mal_id', 'url', 'website_url', 'image_url', 'name', 'given_name', 'family_name', 'birthday', 'favorites', 'relevant_location', 'birth_year']

person_alternate_names — shape=(20408, 2) — 2 colonne
['person_mal_id', 'alt_name']
